# Syn-Chain ABSA — UIT-VSFC (Tiếng Việt)

In [14]:
!pip install -q langchain langchain-openai langchain-core langsmith
!pip install -q underthesea

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 20.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 51.9 MB/s eta 0:00:00


In [15]:
import os
from langchain_openai import ChatOpenAI

QWEN_API_BASE="https://aydin-unshining-nonconstructively.ngrok-free.dev/v1"
QWEN_API_KEY="local-key"
MODEL_NAME="qwen2.5-14b-instruct"

llm = ChatOpenAI(
    model=MODEL_NAME,
    api_key=QWEN_API_KEY,
    base_url=QWEN_API_BASE,
    temperature=0.0,
)
print(f"Khởi tạo LLM: {MODEL_NAME}")

os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGCHAIN_API_KEY'] = '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'
os.environ['LANGCHAIN_PROJECT']    = 'syn-chain-absa_uit-vsfc'

Khởi tạo LLM: qwen2.5-14b-instruct


In [16]:
from underthesea import dependency_parse, pos_tag, word_tokenize

def get_syntactic_dependency_string(text: str) -> str:
    """
    Phân tích cú pháp tiếng Việt bằng underthesea.
    Output: CoNLL-like format (ID, FORM, POS, HEAD_ID, DEPREL)
    """
    result = dependency_parse(text)
    # result là list of (word, head_index, dep_label)
    lines = []
    for i, (word, head, dep) in enumerate(result):
        # Lấy POS tag riêng
        pos_result = pos_tag(word)
        pos = pos_result[0][1] if pos_result else "_"
        fields = [
            str(i + 1),   # ID
            word,          # FORM
            pos,           # POS
            str(head),     # HEAD
            dep,           # DEPREL
        ]
        lines.append("\t".join(fields))
    return "\n".join(lines)

# Test
test_sent = "Dịch vụ rất tệ nhưng đồ ăn thì ngon."
print(f"Test: '{test_sent}'\n")
print(get_syntactic_dependency_string(test_sent))

Test: 'Dịch vụ rất tệ nhưng đồ ăn thì ngon.'

Downloading: "https://github.com/undertheseanlp/underthesea/releases/download/resources/vi-dp-v1a1.zip" to /root/.cache/torch/hub/checkpoints/vi-dp-v1a1.zip


100%|██████████| 45.4M/45.4M [00:01<00:00, 28.5MB/s]
/usr/local/lib/python3.12/dist-packages/torch/hub.py:883: FutureWarning: Falling back to the old format < 1.6. This support will be deprecated in favor of default zipfile format introduced in 1.6. Please redo torch.save() to save it in the new zipfile format.
  return _legacy_zip_load(cached_file, model_dir, map_location, weights_only)


FieldEmbeddings(n_vocab=1000)


1	Dịch vụ	N	3	nsubj
2	rất	R	3	advmod
3	tệ	M	0	root
4	nhưng	C	7	cc
5	đồ ăn	M	7	nsubj
6	thì	C	7	mark
7	ngon	A	3	conj
8	.	CH	3	punct


In [17]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langsmith import traceable

# ---- Bước 1: Phân tích cú pháp ----
prompt1 = ChatPromptTemplate.from_messages([
    ("system", "Bạn là chuyên gia ngôn ngữ học tiếng Việt."),
    ("human", """
        Câu: {text}
        Khía cạnh (Aspect): {aspect}
        
        Chuỗi phụ thuộc cú pháp:
        {dependency_seq}
        
        Giải thích: Mỗi dòng gồm: ID từ | dạng từ | nhãn POS | ID từ cha (head) | quan hệ phụ thuộc.
        
        Hãy tìm trong cây phụ thuộc các từ có quan hệ trực tiếp với "{aspect}" (là head hoặc dependent của nó).
        Liệt kê ngắn gọn: từ nào, quan hệ gì, đóng vai trò gì với "{aspect}".
    """)
])

# ---- Bước 2: Trích xuất quan điểm ----
prompt2 = ChatPromptTemplate.from_messages([
    ("system", "Bạn là chuyên gia trích xuất quan điểm từ văn bản tiếng Việt."),
    ("human", """
        Câu: {text}
        Khía cạnh (Aspect): {aspect}
        
        Phân tích cú pháp: {step1_output}
        
        Dựa vào ngữ cảnh và thông tin cú pháp, người nói có quan điểm như thế nào về "{aspect}"?
        Hãy chỉ ra các từ/cụm từ mô tả cụ thể nếu có.
    """)
])

# ---- Bước 3: Phân loại cảm xúc ----
prompt3 = ChatPromptTemplate.from_messages([
    ("system", "Bạn là chuyên gia phân tích cảm xúc. Chỉ trả lời đúng một từ: Positive, Negative, hoặc Neutral."),
    ("human", """
        Quan điểm: {step2_output}
        
        Dựa trên quan điểm được mô tả về "{aspect}", cảm xúc của người nói là gì?
        Trả lời ĐÚNG MỘT từ trong: Positive, Negative, Neutral.
    """)
])

parser = StrOutputParser()
chain1 = prompt1 | llm | parser
chain2 = prompt2 | llm | parser
chain3 = prompt3 | llm | parser


@traceable(run_type='chain', name='Syn-Chain Vietnamese Pipeline')
def run_syn_chain(text: str, aspect: str, dependency_seq: str) -> dict:
    step1_out = chain1.invoke({"text": text, "aspect": aspect, "dependency_seq": dependency_seq})
    step2_out = chain2.invoke({"text": text, "aspect": aspect, "step1_output": step1_out})
    step3_out = chain3.invoke({"aspect": aspect, "step2_output": step2_out})
    return {
        "step1_syntax":  step1_out,
        "step2_opinion": step2_out,
        "step3_sentiment": step3_out.strip()
    }

print("Pipeline khởi tạo xong.")

Pipeline khởi tạo xong.


In [18]:
import json
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report

def evaluate_syn_chain(
    data,
    limit=None,
    output_log_path="evaluation_logs.json"
):
    if limit:
        data = data[:limit]
        print(f" Giới hạn {limit} ví dụ")

    y_true, y_pred, logs = [], [], []

    for item in tqdm(data, desc="Evaluating"):
        text          = item['text']
        aspect        = item['aspect']
        true_sentiment = item['sentiment'].lower()

        dep_seq = get_syntactic_dependency_string(text)

        try:
            result   = run_syn_chain(text, aspect, dep_seq)
            pred_raw = result['step3_sentiment'].lower()

            if "positive" in pred_raw:
                pred = "positive"
            elif "negative" in pred_raw:
                pred = "negative"
            else:
                pred = "neutral"

            y_true.append(true_sentiment)
            y_pred.append(pred)
            logs.append({
                "text":         text,
                "aspect":       aspect,
                "ground_truth": true_sentiment,
                "prediction":   pred,
                "llm_reasoning": result
            })

        except Exception as e:
            print(f"\nLỗi: '{text[:50]}...' — {e}")
            continue

    if y_true:
        acc = accuracy_score(y_true, y_pred)
        f1  = f1_score(y_true, y_pred, average='macro', zero_division=0)

        print("\n" + "=" * 40)
        print("KẾT QUẢ ĐÁNH GIÁ")
        print("=" * 40)
        print(f"Số mẫu      : {len(y_true)}")
        print(f"Accuracy    : {acc:.4f}")
        print(f"Macro-F1    : {f1:.4f}")
        print()
        print(classification_report(y_true, y_pred, zero_division=0))
        print("=" * 40)

        with open(output_log_path, "w", encoding='utf-8') as f:
            json.dump(logs, f, indent=4, ensure_ascii=False)
        print(f"Đã lưu logs tại: {output_log_path}")
    else:
        print("Không có mẫu nào được đánh giá thành công.")

print("Hàm evaluate sẵn sàng.")

Hàm evaluate sẵn sàng.


In [ ]:
data_path = '/kaggle/input/datasets/duckhoi/dataset-for-survey-synchain/uit-vsfc_test.json'

with open(data_path, 'r', encoding='utf-8') as f:
    test_data = json.load(f)

print(f"Đã load {len(test_data)} mẫu từ: {data_path}")

evaluate_syn_chain(test_data, limit=200)

Đã load 3166 mẫu từ: /kaggle/input/datasets/duckhoi/dataset-for-survey-synchain/uit-vsfc_test.json
 Giới hạn 200 ví dụ


Evaluating:   0%|          | 0/200 [00:00<?, ?it/s]
  0%|                                    | 0/6 00:00<?, ?it/s
Evaluating:   0%|          | 1/200 [00:43<2:23:27, 43.25s/it]
  0%|                                    | 0/5 00:00<?, ?it/s
Evaluating:   1%|          | 2/200 [01:19<2:09:37, 39.28s/it]
  0%|                                    | 0/6 00:00<?, ?it/s
Evaluating:   2%|▏         | 3/200 [02:08<2:23:06, 43.59s/it]
  0%|                                    | 0/7 00:00<?, ?it/s
                                                             